In [0]:
import yaml
from pyspark.sql import functions as F

In [0]:
config_path = "/Workspace/Repos/adb-tetiana@startsteps.org/beiersdorf-live-demo/config/project_config.yml"

In [0]:
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

In [0]:
catalog_name = config["catalog"]
schema_name = config["schema"]

silver_sales_table = f"{catalog_name}.{schema_name}.{config['tables']['silver_sales']}"

print("Silver sales table:", silver_sales_table)

In [0]:
silver_df = spark.table(silver_sales_table)

print("Loaded silver sales table:", silver_sales_table)
display(silver_df)

In [0]:
row_count = silver_df.count()

null_market_count = silver_df.filter(F.col("market").isNull()).count()
null_brand_count = silver_df.filter(F.col("brand").isNull()).count()
negative_sales_count = silver_df.filter(F.col("sales_amount") < 0).count()
null_report_date_count = silver_df.filter(F.col("report_date").isNull()).count()

print("Silver row count:", row_count)
print("Rows with null market:", null_market_count)
print("Rows with null brand:", null_brand_count)
print("Rows with null report_date:", null_report_date_count)
print("Rows with negative sales_amount:", negative_sales_count)


In [0]:
quality_summary_data = [
    ("row_count", row_count),
    ("null_market_count", null_market_count),
    ("null_brand_count", null_brand_count),
    ("null_report_date_count", null_report_date_count),
    ("negative_sales_count", negative_sales_count),
]

In [0]:
quality_summary_df = spark.createDataFrame(
    quality_summary_data,
    ["check_name", "check_value"]
)

In [0]:
display(quality_summary_df)


In [0]:
market_counts_df = (
    silver_df
    .groupBy("market")
    .count()
    .orderBy("market")
)

display(market_counts_df)

In [0]:
brand_counts_df = (
    silver_df
    .groupBy("brand")
    .count()
    .orderBy("brand")
)

display(brand_counts_df)

## Set task values for Databricks Jobs

In [0]:
try:
    dbutils.jobs.taskValues.set(key="silver_row_count", value=row_count)
    dbutils.jobs.taskValues.set(key="null_market_count", value=null_market_count)
    dbutils.jobs.taskValues.set(key="null_brand_count", value=null_brand_count)
    dbutils.jobs.taskValues.set(key="null_report_date_count", value=null_report_date_count)
    dbutils.jobs.taskValues.set(key="negative_sales_count", value=negative_sales_count)

    print("Task values set successfully:")
    print("silver_row_count =", row_count)
    print("null_market_count =", null_market_count)
    print("null_brand_count =", null_brand_count)
    print("null_report_date_count =", null_report_date_count)
    print("negative_sales_count =", negative_sales_count)
except Exception as e:
    print("Task values not set.")
    print("This is normal if the notebook is not running inside a Databricks Job.")
    print("Reason:", e)